In [ ]:
# Load environment variables (ANTHROPIC_API_KEY) from .env
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import json
from dataclasses import dataclass

from utils.chat import Chat

@dataclass
class EvalCase:
    task: str
    format: str
    solution_criteria: str

@dataclass
class Grade:
    score: int
    strengths: list[str]
    weaknesses: list[str]
    reasoning: str

@dataclass
class EvalResult:
    score: float 
    case: EvalCase
    model_output: str
    grade: Grade

In [ ]:
def run_prompt(case: EvalCase) -> str:
    """Builds a prompt from the task and returns the model's raw solution."""
    prompt = f"""
Please, solve the following task:

{case.task}

* Respond only with Python, JSON or a plain Regex
* Do not add any comments or commentary explanation
"""
    return Chat(model="claude-opus-4-5").user(prompt).send(prefill="```code", stop_sequences=["```"])

In [ ]:
def grade_by_model(case: EvalCase, model_output: str) -> Grade:
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{case.task}
</task>

Solution criteria:
<criteria>
{case.solution_criteria}
</criteria>

Solution to Evaluate:
<solution>
{model_output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with these fields:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    output = Chat(model="claude-opus-4-5").user(eval_prompt).send(prefill="```json", stop_sequences=["```"])
    return Grade(**json.loads(output))

In [ ]:
import re
import ast

def is_valid_json(text: str) -> bool:
    try:
        json.loads(text.strip())
        return True
    except json.JSONDecodeError:
        return False

def is_valid_python(text: str) -> bool:
    try:
        ast.parse(text.strip())
        return True
    except SyntaxError:
        return False

def is_valid_regex(text: str) -> bool:
    try:
        re.compile(text.strip())
        return True
    except re.error:
        return False

def is_valid_syntax(case: EvalCase, model_output: str) -> bool:
    validators = {
        "json": is_valid_json,
        "python": is_valid_python,
        "regex": is_valid_regex,
    }
    return validators[case.format](model_output)

In [ ]:
def run_test_case(case: EvalCase) -> EvalResult:
    """Runs the prompt, then grades it — invalid syntax gates the score to 0."""
    model_output = run_prompt(case)

    grade = grade_by_model(case, model_output)
    valid_syntax = is_valid_syntax(case, model_output)

    return EvalResult(
        score=grade.score if valid_syntax else 0,
        case=case,
        model_output=model_output,
        grade=grade)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from statistics import mean

def _grade(case: EvalCase) -> EvalResult | None:
    try:
        return run_test_case(case)
    except Exception as e:
        print(f"Skipped a case: {e}")
        return None

def run_eval(dataset: list[EvalCase], max_workers: int = 3) -> list[EvalResult]:
    """Grades every eval case in parallel and prints the average of the successes."""
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = [r for r in executor.map(_grade, dataset) if r is not None]

    if results:
        print(f"AVG score: {mean(r.score for r in results)}")

    return results

In [ ]:
with open("002_prompt_evals/dataset.json", "r") as f:
    dataset = [EvalCase(**item) for item in json.load(f)]

results = run_eval(dataset)

In [ ]:
from rich import print

print(results)